Inspect every CSV file in a folder

In [3]:
import os
import pandas as pd

# ====== CHANGE THIS TO YOUR FOLDER ======
DATA_DIR = r"/nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes"
# =======================================

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

def find_likely_columns(columns):
    cols_lower = {c.lower(): c for c in columns}

    def match_any(keywords):
        hits = []
        for c in columns:
            c_low = c.lower()
            if any(k in c_low for k in keywords):
                hits.append(c)
        return hits

    return {
        "encounter_cols": match_any(["encounter", "csn", "encounterepiccsn", "epicencountercsn"]),
        "mrn_cols": match_any(["mrn", "patientmrn", "primarymrn"]),
        "note_cols": match_any(["note", "text", "notetext"]),
        "complaint_cols": match_any(["complaint", "chief"]),
        "pe_label_cols": match_any(["pe", "diagnosis", "label"]),
        "time_cols": match_any(["time", "date", "instant", "arrival"]),
    }

def inspect_csv(file_path, nrows_preview=5):
    print("=" * 120)
    print(f"FILE: {os.path.basename(file_path)}")

    try:
        df = pd.read_csv(file_path, low_memory=False)
    except Exception as e:
        print(f"Could not read file: {e}")
        return

    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print("\nColumns:")
    print(list(df.columns))

    likely = find_likely_columns(df.columns)
    print("\nLikely important columns:")
    for k, v in likely.items():
        print(f"  {k}: {v if v else 'None'}")

    print("\nDtypes:")
    print(df.dtypes)

    print("\nMissing values (top 20):")
    missing = df.isna().sum().sort_values(ascending=False)
    print(missing.head(20))

    # Show approximate uniqueness for likely keys
    key_candidates = list(set(
        likely["encounter_cols"] +
        likely["mrn_cols"] +
        likely["complaint_cols"]
    ))

    if key_candidates:
        print("\nUniqueness check for likely key columns:")
        for col in key_candidates:
            try:
                nunique = df[col].nunique(dropna=True)
                print(f"  {col}: {nunique:,} unique / {len(df):,} rows")
            except Exception as e:
                print(f"  {col}: could not compute unique count ({e})")

    print("\nPreview:")
    print(df.head(nrows_preview))

    # Optional: show a few long-text examples if present
    text_candidates = likely["note_cols"]
    if text_candidates:
        print("\nSample text lengths:")
        for col in text_candidates[:3]:
            try:
                lengths = df[col].astype(str).str.len()
                print(
                    f"  {col}: "
                    f"non-null={df[col].notna().sum():,}, "
                    f"mean_len={lengths.mean():.1f}, "
                    f"max_len={lengths.max():.0f}"
                )
            except Exception as e:
                print(f"  {col}: could not summarize text ({e})")

def main():
    csv_files = sorted(
        [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv")]
    )

    print(f"Found {len(csv_files)} CSV files.\n")
    for f in csv_files:
        inspect_csv(f)

if __name__ == "__main__":
    main()

Found 7 CSV files.

FILE: ED-triage-and-provider-sample.csv
Shape: 438 rows x 15 columns

Columns:
['IndexEncounterKey', 'IndexEncounterCsn', 'PatientDurableKey', 'PatientMrn', 'IndexArrivalInstant', 'ChiefComplaint', 'ClinicalNoteKey', 'Service', 'Type', 'AuthorType', 'ServiceInstant', 'CreationInstant', 'LastEditedInstant', 'Status', 'NoteText']

Likely important columns:
  encounter_cols: ['IndexEncounterKey', 'IndexEncounterCsn']
  mrn_cols: ['PatientMrn']
  note_cols: ['ClinicalNoteKey', 'NoteText']
  complaint_cols: ['ChiefComplaint']
  pe_label_cols: ['Type', 'AuthorType']
  time_cols: ['IndexArrivalInstant', 'ServiceInstant', 'CreationInstant', 'LastEditedInstant']

Dtypes:
IndexEncounterKey       int64
IndexEncounterCsn       int64
PatientDurableKey       int64
PatientMrn              int64
IndexArrivalInstant    object
ChiefComplaint         object
ClinicalNoteKey         int64
Service                object
Type                   object
AuthorType             object
ServiceIn

Create cross walk files

In [11]:
#!/usr/bin/env python3
"""
Build 4 project tables for PE / chest-pain-SOB / CT / notes work:

1) encounter_master.csv / .parquet
   - one row per encounter
   - encounter-level registry / crosswalk
   - includes cohort flags, counts, timing, overlap/sample flags, etc.

2) notes_raw.csv / .parquet
   - one row per raw text item
   - provider notes + triage notes + CT narratives + CT impressions

3) encounter_text_aggregates.csv / .parquet
   - one row per encounter
   - concatenated text grouped by note source/channel
   - ready for prompting / summarization

4) summary_outputs_template.jsonl
   - one JSON object per encounter
   - blank schema for future structured summarization output
   - includes metadata fields for guideline/prompt/model versioning

Key design choices:
- Encounter is the anchor unit.
- Triage / provider / CT are separate text channels under the broader text modality.
- CP/SOB notes file includes both provider and triage notes; note_source is derived from Type.
- PE cohort flags represent suspected PE / PE diagnostic consideration,
  not confirmed PE diagnosis.
- ED-triage-and-provider-sample is optional and used only for overlap/reference flags.
- note_id is treated as STRING everywhere so parquet writes safely.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


# =============================================================================
# CONFIG
# =============================================================================

DATA_DIR = Path(r"/nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes")
OUTPUT_DIR = DATA_DIR / "derived_tables"

# Main inputs
FILE_PE_CASE_LIST = "medic-saf-pe-cases-mich-med-2023-2025-case-list.csv"
FILE_CP_SOB_CASE_LIST = "mich-med-chest-pain-sob-visits-2023-2025-case-list.csv"
FILE_PE_PROVIDER_NOTES = "medic-saf-pe-cases-mich-med-2023-2025-case-ed-provider-notes.csv"
FILE_CP_SOB_PROVIDER_TRIAGE_NOTES = "mich-med-chest-pain-sob-visits-2023-2025-case-ed-provider-and-triage-notes.csv"
FILE_CT_REPORTS = "mich-med-ct-chest-2023-2025-reports.csv"
FILE_LABELED_200 = "notes-for-200-cases.csv"

# Optional reference-only sample file
FILE_ED_TRIAGE_PROVIDER_SAMPLE = "ED-triage-and-provider-sample.csv"

# Whether to inspect overlap with ED-triage-and-provider-sample
CHECK_SAMPLE_OVERLAP = True

# Whether to attempt datetime parsing on known time columns
PARSE_DATETIMES = True

# Whether to include full CT narrative text in notes_raw and aggregates
INCLUDE_CT_NARRATIVE = True

# Whether to also write parquet versions
WRITE_PARQUET = True

# Text separator used for encounter-level concatenation
TEXT_SEP = "\n\n" + ("=" * 80) + "\n\n"


# =============================================================================
# UTILITIES
# =============================================================================

def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def read_csv_required(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    print(f"Reading: {path.name}")
    return pd.read_csv(path, low_memory=False)


def normalize_mrn_series(s: pd.Series) -> pd.Series:
    """
    Normalize MRN-like values to string with leading zeros preserved if present.
    Uses 9-digit zero-padding for numeric values when possible, matching SQL style.
    """
    out = s.copy().astype("string").str.strip()
    out = out.str.replace(r"\.0$", "", regex=True)
    is_digits = out.fillna("").str.fullmatch(r"\d+")
    out = out.where(~is_digits, out.str.zfill(9))
    return out


def safe_to_datetime(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df


def coalesce_columns(df: pd.DataFrame, output_col: str, candidate_cols: list[str]) -> pd.DataFrame:
    existing = [c for c in candidate_cols if c in df.columns]
    if not existing:
        df[output_col] = pd.NA
        return df

    out = pd.Series([pd.NA] * len(df), index=df.index, dtype="object")
    for c in existing:
        out = out.fillna(df[c])
    df[output_col] = out
    return df


def text_len_series(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.len()


def first_non_null(series: pd.Series) -> Any:
    x = series.dropna()
    return x.iloc[0] if len(x) else pd.NA


def mode_or_first(series: pd.Series) -> Any:
    x = series.dropna()
    if len(x) == 0:
        return pd.NA
    m = x.mode(dropna=True)
    return m.iloc[0] if len(m) else x.iloc[0]


def unique_non_null_join(series: pd.Series, sep: str = " | ") -> str | None:
    vals = [str(v) for v in series.dropna().astype(str).unique().tolist() if str(v).strip() != ""]
    if not vals:
        return None
    return sep.join(vals)


def detect_ct_pe_protocol(study_name: pd.Series, cpt: pd.Series) -> pd.Series:
    """
    Heuristic flag for PE-oriented CT chest protocol.
    Uses study name text and common CPT 71275.
    """
    study = study_name.fillna("").astype(str).str.upper()
    cpt_str = cpt.astype("string").fillna("")
    return (
        study.str.contains(r"\bPE\b", regex=True)
        | study.str.contains("ANGIO CHEST", regex=False)
        | study.str.contains("CT ANGIO CHEST", regex=False)
        | (cpt_str == "71275")
    ).astype(int)


def complaint_flags_from_text(s: pd.Series) -> pd.DataFrame:
    txt = s.fillna("").astype(str).str.upper()
    return pd.DataFrame({
        "is_chest_pain": txt.str.contains("CHEST PAIN", regex=False).astype(int),
        "is_shortness_of_breath": txt.str.contains("SHORTNESS OF BREATH", regex=False).astype(int),
    })


def clean_text_for_concat(s: pd.Series) -> pd.Series:
    return (
        s.fillna("")
         .astype(str)
         .str.replace(r"\r\n?", "\n", regex=True)
         .str.replace(r"[ \t]+", " ", regex=True)
         .str.strip()
    )


def prepend_text_header(source: str, subtype: str | None, ts: Any, note_id: Any, text: str) -> str:
    parts = [f"SOURCE={source}"]
    if subtype is not None and str(subtype) != "":
        parts.append(f"SUBTYPE={subtype}")
    if pd.notna(ts):
        parts.append(f"TIME={ts}")
    if pd.notna(note_id):
        parts.append(f"NOTE_ID={note_id}")
    header = " | ".join(parts)
    return f"[{header}]\n{text}"


def make_columns_parquet_safe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert object columns to pandas string dtype so pyarrow doesn't choke on mixed object types.
    """
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_object_dtype(df[c]):
            df[c] = df[c].astype("string")
    return df


# =============================================================================
# LOAD INPUTS
# =============================================================================

ensure_dir(OUTPUT_DIR)

pe_case_list = read_csv_required(DATA_DIR / FILE_PE_CASE_LIST)
cp_sob_case_list = read_csv_required(DATA_DIR / FILE_CP_SOB_CASE_LIST)
pe_provider_notes = read_csv_required(DATA_DIR / FILE_PE_PROVIDER_NOTES)
cp_sob_notes_all = read_csv_required(DATA_DIR / FILE_CP_SOB_PROVIDER_TRIAGE_NOTES)
ct_reports = read_csv_required(DATA_DIR / FILE_CT_REPORTS)
labeled_200 = read_csv_required(DATA_DIR / FILE_LABELED_200)

sample_df = None
if CHECK_SAMPLE_OVERLAP and (DATA_DIR / FILE_ED_TRIAGE_PROVIDER_SAMPLE).exists():
    sample_df = read_csv_required(DATA_DIR / FILE_ED_TRIAGE_PROVIDER_SAMPLE)

# =============================================================================
# STANDARDIZE CASE LISTS
# =============================================================================

pe_enc = pe_case_list.rename(columns={
    "EpicEncounterCSN": "EncounterCsn",
    "PatientMrn": "PatientMrn",
}).copy()

pe_enc["EncounterCsn"] = pd.to_numeric(pe_enc["EncounterCsn"], errors="coerce").astype("Int64")
pe_enc["PatientMrn"] = normalize_mrn_series(pe_enc["PatientMrn"])
pe_enc["in_pe_workup_cohort"] = 1
pe_enc["pe_suspected"] = 1

cp_sob_enc = cp_sob_case_list.rename(columns={
    "EncounterEpicCsn": "EncounterCsn",
    "PrimaryMrn": "PatientMrn",
    "Name": "ChiefComplaint",
}).copy()

cp_sob_enc["EncounterCsn"] = pd.to_numeric(cp_sob_enc["EncounterCsn"], errors="coerce").astype("Int64")
cp_sob_enc["PatientMrn"] = normalize_mrn_series(cp_sob_enc["PatientMrn"])
cp_sob_enc["ChiefComplaint"] = cp_sob_enc["ChiefComplaint"].astype("string")
cp_sob_enc["in_chest_pain_sob_cohort"] = 1

cp_flags = complaint_flags_from_text(cp_sob_enc["ChiefComplaint"])
cp_sob_enc = pd.concat([cp_sob_enc, cp_flags], axis=1)

# =============================================================================
# STANDARDIZE NOTES
# =============================================================================

def prepare_notes(df: pd.DataFrame, source_dataset: str) -> pd.DataFrame:
    out = df.rename(columns={
        "IndexEncounterCsn": "EncounterCsn",
        "IndexArrivalInstant": "ArrivalInstant",
    }).copy()

    out["EncounterCsn"] = pd.to_numeric(out["EncounterCsn"], errors="coerce").astype("Int64")
    out["PatientMrn"] = normalize_mrn_series(out["PatientMrn"])

    if PARSE_DATETIMES:
        out = safe_to_datetime(out, [
            "ArrivalInstant", "ServiceInstant", "CreationInstant", "LastEditedInstant",
            "NoteEncounterArrivalInstant"
        ])

    type_series = out["Type"].astype("string").str.upper()
    out["note_source"] = np.where(
        type_series.str.contains("TRIAGE", na=False),
        "triage",
        "provider"
    )

    out["note_subtype"] = out["Type"].astype("string")
    out["note_status"] = out["Status"].astype("string")
    out["note_author_type"] = out["AuthorType"].astype("string")
    out["note_service"] = out["Service"].astype("string")
    out["source_dataset"] = source_dataset

    out = coalesce_columns(out, "note_time", ["ServiceInstant", "CreationInstant", "LastEditedInstant"])
    text_col = "Text" if "Text" in out.columns else "NoteText"
    out["text"] = clean_text_for_concat(out[text_col])

    out["note_id"] = out["ClinicalNoteKey"].astype("string")
    out["text_length"] = text_len_series(out["text"]).astype("Int64")

    if "IsIndexEncounterNote" in out.columns:
        out["is_index_encounter_note"] = pd.to_numeric(
            out["IsIndexEncounterNote"], errors="coerce"
        ).fillna(0).astype("Int64")
    else:
        out["is_index_encounter_note"] = pd.Series(pd.NA, index=out.index, dtype="Int64")

    out["ChiefComplaint"] = out["ChiefComplaint"].astype("string")

    keep_cols = [
        "EncounterCsn",
        "PatientMrn",
        "ArrivalInstant",
        "ChiefComplaint",
        "note_id",
        "note_source",
        "note_subtype",
        "note_status",
        "note_author_type",
        "note_service",
        "note_time",
        "text",
        "text_length",
        "source_dataset",
        "ClinicalNoteKey",
        "ServiceInstant",
        "CreationInstant",
        "LastEditedInstant",
        "is_index_encounter_note",
        "NoteEncounterCsn",
        "NoteEncounterArrivalInstant",
    ]
    keep_cols = [c for c in keep_cols if c in out.columns]
    return out[keep_cols].copy()


provider_pe = prepare_notes(pe_provider_notes, "pe_provider_notes")
notes_cp_sob = prepare_notes(cp_sob_notes_all, "cp_sob_provider_and_triage_notes")

notes_all = pd.concat([provider_pe, notes_cp_sob], ignore_index=True)

notes_all = (
    notes_all
    .sort_values(["EncounterCsn", "note_id", "source_dataset"])
    .drop_duplicates(subset=["EncounterCsn", "note_id"], keep="first")
    .reset_index(drop=True)
)

provider_df = notes_all[notes_all["note_source"] == "provider"].copy()
triage_df = notes_all[notes_all["note_source"] == "triage"].copy()

provider_summary = (
    provider_df
    .groupby("EncounterCsn", dropna=False)
    .agg(
        PatientMrn_provider=("PatientMrn", first_non_null),
        provider_note_count=("note_id", "nunique"),
        provider_note_row_count=("note_id", "size"),
        has_provider_note=("note_id", lambda s: int(s.notna().any())),
        first_provider_note_time=("note_time", "min"),
        last_provider_note_time=("note_time", "max"),
        provider_text_char_count=("text_length", "sum"),
        provider_max_note_char_count=("text_length", "max"),
        provider_note_types=("note_subtype", unique_non_null_join),
        provider_author_types=("note_author_type", unique_non_null_join),
        provider_services=("note_service", unique_non_null_join),
        provider_note_statuses=("note_status", unique_non_null_join),
        provider_source_datasets=("source_dataset", unique_non_null_join),
        arrival_from_provider_notes=("ArrivalInstant", first_non_null),
        chief_complaint_from_provider_notes=("ChiefComplaint", mode_or_first),
    )
    .reset_index()
)

provider_summary["multiple_provider_notes"] = (
    provider_summary["provider_note_count"] > 1
).astype(int)

triage_summary = (
    triage_df
    .groupby("EncounterCsn", dropna=False)
    .agg(
        PatientMrn_triage=("PatientMrn", first_non_null),
        triage_note_count=("note_id", "nunique"),
        triage_note_row_count=("note_id", "size"),
        has_triage_note=("note_id", lambda s: int(s.notna().any())),
        first_triage_note_time=("note_time", "min"),
        last_triage_note_time=("note_time", "max"),
        triage_text_char_count=("text_length", "sum"),
        triage_max_note_char_count=("text_length", "max"),
        triage_note_types=("note_subtype", unique_non_null_join),
        triage_author_types=("note_author_type", unique_non_null_join),
        triage_services=("note_service", unique_non_null_join),
        triage_note_statuses=("note_status", unique_non_null_join),
        arrival_from_triage_notes=("ArrivalInstant", first_non_null),
        chief_complaint_from_triage_notes=("ChiefComplaint", mode_or_first),
    )
    .reset_index()
)

# =============================================================================
# CT REPORTS -> RAW TEXT TABLE + ENCOUNTER SUMMARY
# =============================================================================

ct = ct_reports.rename(columns={
    "EpicEncounterCsn": "EncounterCsn",
    "MedicArrivalInstant": "ArrivalInstant",
}).copy()

ct["EncounterCsn"] = pd.to_numeric(ct["EncounterCsn"], errors="coerce").astype("Int64")

if PARSE_DATETIMES:
    ct = safe_to_datetime(ct, ["ArrivalInstant", "ImagingOrderedInstant", "ImagingPerformedInstant"])

ct["ChiefComplaint"] = ct["ChiefComplaint"].astype("string")
ct["StudyName"] = ct["StudyName"].astype("string")
ct["CPT"] = ct["CPT"].astype("string")
ct["ImagingNarrative"] = clean_text_for_concat(ct["ImagingNarrative"].astype("string"))
ct["ImagingImpression"] = clean_text_for_concat(ct["ImagingImpression"].astype("string"))
ct["ct_pe_protocol_flag"] = detect_ct_pe_protocol(ct["StudyName"], ct["CPT"])

ct = ct.reset_index(drop=True)
ct["ct_row_id"] = (ct.index + 1).astype("Int64")

ct_narr_raw = ct[[
    "EncounterCsn", "ArrivalInstant", "ChiefComplaint", "StudyName", "CPT",
    "ImagingOrderedInstant", "ImagingPerformedInstant", "ImagingNarrative", "ct_row_id"
]].copy()
ct_narr_raw = ct_narr_raw.rename(columns={"ImagingNarrative": "text"})
ct_narr_raw["PatientMrn"] = pd.NA
ct_narr_raw["note_id"] = ct_narr_raw["ct_row_id"].astype("string") + "_NARR"
ct_narr_raw["note_source"] = "ct_report"
ct_narr_raw["note_subtype"] = "ct_narrative"
ct_narr_raw["note_status"] = pd.NA
ct_narr_raw["note_author_type"] = "radiology"
ct_narr_raw["note_service"] = "radiology"
ct_narr_raw["note_time"] = ct_narr_raw["ImagingPerformedInstant"]
ct_narr_raw["source_dataset"] = "ct_reports"
ct_narr_raw["text_length"] = text_len_series(ct_narr_raw["text"]).astype("Int64")

ct_impr_raw = ct[[
    "EncounterCsn", "ArrivalInstant", "ChiefComplaint", "StudyName", "CPT",
    "ImagingOrderedInstant", "ImagingPerformedInstant", "ImagingImpression", "ct_row_id"
]].copy()
ct_impr_raw = ct_impr_raw.rename(columns={"ImagingImpression": "text"})
ct_impr_raw["PatientMrn"] = pd.NA
ct_impr_raw["note_id"] = ct_impr_raw["ct_row_id"].astype("string") + "_IMPR"
ct_impr_raw["note_source"] = "ct_report"
ct_impr_raw["note_subtype"] = "ct_impression"
ct_impr_raw["note_status"] = pd.NA
ct_impr_raw["note_author_type"] = "radiology"
ct_impr_raw["note_service"] = "radiology"
ct_impr_raw["note_time"] = ct_impr_raw["ImagingPerformedInstant"]
ct_impr_raw["source_dataset"] = "ct_reports"
ct_impr_raw["text_length"] = text_len_series(ct_impr_raw["text"]).astype("Int64")

ct_note_frames = [ct_impr_raw]
if INCLUDE_CT_NARRATIVE:
    ct_note_frames.insert(0, ct_narr_raw)

ct_raw = pd.concat(ct_note_frames, ignore_index=True)
ct_raw = ct_raw[ct_raw["text"].fillna("").astype(str).str.strip() != ""].copy()

ct_raw_keep = [
    "EncounterCsn",
    "PatientMrn",
    "ArrivalInstant",
    "ChiefComplaint",
    "note_id",
    "note_source",
    "note_subtype",
    "note_status",
    "note_author_type",
    "note_service",
    "note_time",
    "text",
    "text_length",
    "source_dataset",
    "StudyName",
    "CPT",
    "ImagingOrderedInstant",
    "ImagingPerformedInstant",
]
ct_raw = ct_raw[ct_raw_keep].copy()

ct_summary = (
    ct
    .groupby("EncounterCsn", dropna=False)
    .agg(
        ct_report_count=("ct_row_id", "size"),
        has_ct_report=("ct_row_id", lambda s: int(s.notna().any())),
        first_ct_ordered_time=("ImagingOrderedInstant", "min"),
        first_ct_time=("ImagingPerformedInstant", "min"),
        last_ct_time=("ImagingPerformedInstant", "max"),
        ct_first_arrival_instant=("ArrivalInstant", "first"),
        ct_primary_chief_complaint=("ChiefComplaint", mode_or_first),
        ct_study_names=("StudyName", unique_non_null_join),
        ct_cpt_codes=("CPT", unique_non_null_join),
        ct_pe_protocol_any=("ct_pe_protocol_flag", "max"),
        ct_narrative_char_count=("ImagingNarrative", lambda s: text_len_series(s).sum()),
        ct_impression_char_count=("ImagingImpression", lambda s: text_len_series(s).sum()),
        ct_max_impression_char_count=("ImagingImpression", lambda s: text_len_series(s).max()),
    )
    .reset_index()
)

# =============================================================================
# LABELED 200
# =============================================================================

lab200 = labeled_200.rename(columns={
    "EncounterCsn": "EncounterCsn",
    "Note_Count": "labeled_200_note_count",
    "Text": "labeled_200_text",
}).copy()

lab200["EncounterCsn"] = pd.to_numeric(lab200["EncounterCsn"], errors="coerce").astype("Int64")
lab200["included_in_labeled_200"] = 1
lab200["labeled_200_text"] = clean_text_for_concat(lab200["labeled_200_text"])
lab200["labeled_200_text_char_count"] = text_len_series(lab200["labeled_200_text"]).astype("Int64")

lab200_summary = lab200[[
    "EncounterCsn",
    "included_in_labeled_200",
    "labeled_200_note_count",
    "labeled_200_text_char_count",
]].copy()

# =============================================================================
# OPTIONAL: SAMPLE OVERLAP FLAGS
# =============================================================================

sample_enc_summary = None

if sample_df is not None:
    samp = sample_df.rename(columns={
        "IndexEncounterCsn": "EncounterCsn",
        "IndexArrivalInstant": "ArrivalInstant",
        "NoteText": "text",
        "Type": "note_subtype",
        "Status": "note_status",
        "AuthorType": "note_author_type",
        "Service": "note_service",
        "ClinicalNoteKey": "note_id",
    }).copy()

    samp["EncounterCsn"] = pd.to_numeric(samp["EncounterCsn"], errors="coerce").astype("Int64")
    samp["PatientMrn"] = normalize_mrn_series(samp["PatientMrn"])
    samp["note_id"] = samp["note_id"].astype("string")

    if PARSE_DATETIMES:
        samp = safe_to_datetime(samp, ["ArrivalInstant", "ServiceInstant", "CreationInstant", "LastEditedInstant"])

    samp["note_source"] = np.where(
        samp["note_subtype"].astype(str).str.contains("TRIAGE", case=False, na=False),
        "triage",
        "provider"
    )
    samp = coalesce_columns(samp, "note_time", ["ServiceInstant", "CreationInstant", "LastEditedInstant"])
    samp["text"] = clean_text_for_concat(samp["text"])
    samp["text_length"] = text_len_series(samp["text"]).astype("Int64")

    sample_enc_summary = (
        samp.groupby("EncounterCsn", dropna=False)
        .agg(
            in_ed_triage_provider_sample=("EncounterCsn", lambda s: 1),
            sample_patient_mrn=("PatientMrn", first_non_null),
            sample_arrival_instant=("ArrivalInstant", first_non_null),
            sample_has_triage_note=("note_source", lambda s: int((s == "triage").any())),
            sample_has_provider_note=("note_source", lambda s: int((s == "provider").any())),
            sample_note_count=("note_id", "nunique"),
        )
        .reset_index()
    )

# =============================================================================
# 2) BUILD notes_raw.csv
# =============================================================================

notes_raw = pd.concat([notes_all, ct_raw], ignore_index=True, sort=False)
notes_raw = notes_raw.sort_values(
    ["EncounterCsn", "note_time", "note_source", "note_id"], na_position="last"
).reset_index(drop=True)

notes_raw["note_order_within_encounter"] = notes_raw.groupby("EncounterCsn").cumcount() + 1
notes_raw["note_order_within_source"] = notes_raw.groupby(["EncounterCsn", "note_source"]).cumcount() + 1

notes_raw["is_triage_note"] = (notes_raw["note_source"] == "triage").astype(int)
notes_raw["is_provider_note"] = (notes_raw["note_source"] == "provider").astype(int)
notes_raw["is_ct_note"] = (notes_raw["note_source"] == "ct_report").astype(int)
notes_raw["is_ct_impression"] = (
    (notes_raw["note_source"] == "ct_report") &
    (notes_raw["note_subtype"] == "ct_impression")
).astype(int)
notes_raw["is_ct_narrative"] = (
    (notes_raw["note_source"] == "ct_report") &
    (notes_raw["note_subtype"] == "ct_narrative")
).astype(int)

notes_raw_cols = [
    "EncounterCsn",
    "PatientMrn",
    "ArrivalInstant",
    "ChiefComplaint",
    "note_id",
    "note_source",
    "note_subtype",
    "note_status",
    "note_author_type",
    "note_service",
    "note_time",
    "note_order_within_encounter",
    "note_order_within_source",
    "is_triage_note",
    "is_provider_note",
    "is_ct_note",
    "is_ct_impression",
    "is_ct_narrative",
    "text_length",
    "text",
    "source_dataset",
    "ClinicalNoteKey",
    "ServiceInstant",
    "CreationInstant",
    "LastEditedInstant",
    "is_index_encounter_note",
    "NoteEncounterCsn",
    "NoteEncounterArrivalInstant",
    "StudyName",
    "CPT",
    "ImagingOrderedInstant",
    "ImagingPerformedInstant",
]
notes_raw_cols = [c for c in notes_raw_cols if c in notes_raw.columns]
notes_raw = notes_raw[notes_raw_cols].copy()

notes_raw_string_cols = [
    "PatientMrn",
    "ChiefComplaint",
    "note_id",
    "note_source",
    "note_subtype",
    "note_status",
    "note_author_type",
    "note_service",
    "text",
    "source_dataset",
    "StudyName",
    "CPT",
]
for c in notes_raw_string_cols:
    if c in notes_raw.columns:
        notes_raw[c] = notes_raw[c].astype("string")

# =============================================================================
# 3) BUILD encounter_text_aggregates.csv
# =============================================================================

agg_input = notes_raw.copy()
agg_input["text"] = clean_text_for_concat(agg_input["text"])

agg_input["text_with_header"] = agg_input.apply(
    lambda row: prepend_text_header(
        source=row.get("note_source"),
        subtype=row.get("note_subtype"),
        ts=row.get("note_time"),
        note_id=row.get("note_id"),
        text=row.get("text", ""),
    ),
    axis=1
)

encounter_all_text = (
    agg_input.sort_values(["EncounterCsn", "note_time", "note_source", "note_id"], na_position="last")
    .groupby("EncounterCsn", dropna=False)["text_with_header"]
    .apply(lambda s: TEXT_SEP.join([x for x in s if str(x).strip() != ""]))
    .reset_index(name="all_text_concat")
)

def aggregate_channel(df: pd.DataFrame, source: str, out_col: str) -> pd.DataFrame:
    sub = df[df["note_source"] == source].copy()
    if sub.empty:
        return pd.DataFrame(columns=["EncounterCsn", out_col])
    return (
        sub.sort_values(["EncounterCsn", "note_time", "note_id"], na_position="last")
        .groupby("EncounterCsn", dropna=False)["text_with_header"]
        .apply(lambda s: TEXT_SEP.join([x for x in s if str(x).strip() != ""]))
        .reset_index(name=out_col)
    )

def aggregate_subtype(df: pd.DataFrame, subtype: str, out_col: str) -> pd.DataFrame:
    sub = df[df["note_subtype"] == subtype].copy()
    if sub.empty:
        return pd.DataFrame(columns=["EncounterCsn", out_col])
    return (
        sub.sort_values(["EncounterCsn", "note_time", "note_id"], na_position="last")
        .groupby("EncounterCsn", dropna=False)["text_with_header"]
        .apply(lambda s: TEXT_SEP.join([x for x in s if str(x).strip() != ""]))
        .reset_index(name=out_col)
    )

enc_text_provider = aggregate_channel(agg_input, "provider", "provider_text_concat")
enc_text_triage = aggregate_channel(agg_input, "triage", "triage_text_concat")
enc_text_ct_all = aggregate_channel(agg_input, "ct_report", "ct_text_concat")
enc_text_ct_narr = aggregate_subtype(agg_input, "ct_narrative", "ct_narrative_concat")
enc_text_ct_impr = aggregate_subtype(agg_input, "ct_impression", "ct_impression_concat")

earliest_note_per_source = (
    agg_input.sort_values(["EncounterCsn", "note_source", "note_time", "note_id"], na_position="last")
    .groupby(["EncounterCsn", "note_source"], dropna=False)
    .first()
    .reset_index()
)

earliest_provider = earliest_note_per_source[
    earliest_note_per_source["note_source"] == "provider"
][["EncounterCsn", "text"]].rename(columns={"text": "earliest_provider_note_text"})

earliest_triage = earliest_note_per_source[
    earliest_note_per_source["note_source"] == "triage"
][["EncounterCsn", "text"]].rename(columns={"text": "earliest_triage_note_text"})

earliest_ct = earliest_note_per_source[
    earliest_note_per_source["note_source"] == "ct_report"
][["EncounterCsn", "text"]].rename(columns={"text": "earliest_ct_text"})

encounter_text_aggregates = encounter_all_text.copy()
for extra_df in [
    enc_text_provider,
    enc_text_triage,
    enc_text_ct_all,
    enc_text_ct_narr,
    enc_text_ct_impr,
    earliest_provider,
    earliest_triage,
    earliest_ct,
]:
    encounter_text_aggregates = encounter_text_aggregates.merge(extra_df, on="EncounterCsn", how="left")

for col in [
    "all_text_concat",
    "provider_text_concat",
    "triage_text_concat",
    "ct_text_concat",
    "ct_narrative_concat",
    "ct_impression_concat",
]:
    if col in encounter_text_aggregates.columns:
        encounter_text_aggregates[f"{col}_char_count"] = text_len_series(
            encounter_text_aggregates[col]
        ).astype("Int64")

# =============================================================================
# 1) BUILD encounter_master.csv
# =============================================================================

encounter_universe_parts = [
    pe_enc[["EncounterCsn", "PatientMrn"]],
    cp_sob_enc[["EncounterCsn", "PatientMrn"]],
    provider_summary[["EncounterCsn", "PatientMrn_provider"]].rename(
        columns={"PatientMrn_provider": "PatientMrn"}
    ),
    triage_summary[["EncounterCsn", "PatientMrn_triage"]].rename(
        columns={"PatientMrn_triage": "PatientMrn"}
    ),
    ct[["EncounterCsn"]].assign(PatientMrn=pd.NA),
    lab200[["EncounterCsn"]].assign(PatientMrn=pd.NA),
]
if sample_enc_summary is not None:
    encounter_universe_parts.append(sample_enc_summary[["EncounterCsn"]].assign(PatientMrn=pd.NA))

encounter_universe = pd.concat(encounter_universe_parts, ignore_index=True)
encounter_universe["EncounterCsn"] = pd.to_numeric(encounter_universe["EncounterCsn"], errors="coerce").astype("Int64")
encounter_universe["PatientMrn"] = normalize_mrn_series(encounter_universe["PatientMrn"])
encounter_universe = encounter_universe.drop_duplicates(subset=["EncounterCsn", "PatientMrn"])

encounter_master = (
    encounter_universe
    .groupby("EncounterCsn", dropna=False)
    .agg(PatientMrn=("PatientMrn", first_non_null))
    .reset_index()
)

merge_dfs = [
    cp_sob_enc[[
        "EncounterCsn", "PatientMrn", "ChiefComplaint",
        "in_chest_pain_sob_cohort", "is_chest_pain", "is_shortness_of_breath"
    ]].rename(columns={
        "PatientMrn": "PatientMrn_cp_sob",
        "ChiefComplaint": "chief_complaint_from_case_list"
    }),
    pe_enc[[
        "EncounterCsn", "PatientMrn", "in_pe_workup_cohort", "pe_suspected"
    ]].rename(columns={
        "PatientMrn": "PatientMrn_pe"
    }),
    provider_summary.copy(),
    triage_summary.copy(),
    ct_summary.copy(),
    lab200_summary.copy(),
]

if sample_enc_summary is not None:
    merge_dfs.append(sample_enc_summary.copy())

for mdf in merge_dfs:
    encounter_master = encounter_master.merge(mdf, on="EncounterCsn", how="left")

encounter_master = coalesce_columns(
    encounter_master,
    "PatientMrn",
    ["PatientMrn", "PatientMrn_cp_sob", "PatientMrn_pe", "PatientMrn_provider", "PatientMrn_triage", "sample_patient_mrn"]
)
encounter_master["PatientMrn"] = normalize_mrn_series(encounter_master["PatientMrn"])

encounter_master = coalesce_columns(
    encounter_master,
    "ArrivalInstant",
    ["arrival_from_triage_notes", "arrival_from_provider_notes", "ct_first_arrival_instant", "sample_arrival_instant"]
)

encounter_master = coalesce_columns(
    encounter_master,
    "ChiefComplaint",
    [
        "chief_complaint_from_case_list",
        "chief_complaint_from_triage_notes",
        "chief_complaint_from_provider_notes",
        "ct_primary_chief_complaint",
    ]
)

flag_cols = [
    "in_chest_pain_sob_cohort",
    "is_chest_pain",
    "is_shortness_of_breath",
    "in_pe_workup_cohort",
    "pe_suspected",
    "has_provider_note",
    "has_triage_note",
    "has_ct_report",
    "included_in_labeled_200",
]
if sample_enc_summary is not None:
    flag_cols.extend([
        "in_ed_triage_provider_sample",
        "sample_has_triage_note",
        "sample_has_provider_note",
    ])

for c in flag_cols:
    if c in encounter_master.columns:
        encounter_master[c] = encounter_master[c].fillna(0).astype("Int64")

count_cols = [
    "provider_note_count",
    "provider_note_row_count",
    "triage_note_count",
    "triage_note_row_count",
    "ct_report_count",
    "labeled_200_note_count",
    "labeled_200_text_char_count",
    "provider_text_char_count",
    "provider_max_note_char_count",
    "triage_text_char_count",
    "triage_max_note_char_count",
    "ct_narrative_char_count",
    "ct_impression_char_count",
    "ct_max_impression_char_count",
]
if sample_enc_summary is not None:
    count_cols.append("sample_note_count")

for c in count_cols:
    if c in encounter_master.columns:
        encounter_master[c] = encounter_master[c].fillna(0).astype("Int64")

encounter_master["has_any_text"] = (
    (encounter_master.get("has_provider_note", 0).fillna(0).astype(int) > 0) |
    (encounter_master.get("has_triage_note", 0).fillna(0).astype(int) > 0) |
    (encounter_master.get("has_ct_report", 0).fillna(0).astype(int) > 0)
).astype("Int64")

encounter_master["has_both_provider_and_ct"] = (
    (encounter_master.get("has_provider_note", 0).fillna(0).astype(int) > 0) &
    (encounter_master.get("has_ct_report", 0).fillna(0).astype(int) > 0)
).astype("Int64")

encounter_master["has_any_triage_provider_ct"] = (
    (encounter_master.get("has_triage_note", 0).fillna(0).astype(int) > 0) |
    (encounter_master.get("has_provider_note", 0).fillna(0).astype(int) > 0) |
    (encounter_master.get("has_ct_report", 0).fillna(0).astype(int) > 0)
).astype("Int64")

if "ct_pe_protocol_any" in encounter_master.columns:
    encounter_master["ct_pe_protocol_any"] = encounter_master["ct_pe_protocol_any"].fillna(0).astype("Int64")

comp_fallback = complaint_flags_from_text(encounter_master["ChiefComplaint"])
for col in ["is_chest_pain", "is_shortness_of_breath"]:
    encounter_master[col] = encounter_master[col].fillna(comp_fallback[col]).astype("Int64")

if PARSE_DATETIMES:
    time_cols = [
        "ArrivalInstant",
        "first_provider_note_time", "last_provider_note_time",
        "first_triage_note_time", "last_triage_note_time",
        "first_ct_ordered_time", "first_ct_time", "last_ct_time",
    ]
    if sample_enc_summary is not None:
        time_cols += ["sample_arrival_instant"]

    encounter_master = safe_to_datetime(
        encounter_master,
        [c for c in time_cols if c in encounter_master.columns]
    )

    def diff_minutes(later_col: str, earlier_col: str, out_col: str) -> None:
        if later_col in encounter_master.columns and earlier_col in encounter_master.columns:
            encounter_master[out_col] = (
                (encounter_master[later_col] - encounter_master[earlier_col]).dt.total_seconds() / 60.0
            )

    diff_minutes("first_provider_note_time", "ArrivalInstant", "minutes_from_arrival_to_first_provider_note")
    diff_minutes("first_triage_note_time", "ArrivalInstant", "minutes_from_arrival_to_first_triage_note")
    diff_minutes("first_ct_ordered_time", "ArrivalInstant", "minutes_from_arrival_to_first_ct_order")
    diff_minutes("first_ct_time", "ArrivalInstant", "minutes_from_arrival_to_first_ct")
    diff_minutes("first_ct_time", "first_provider_note_time", "minutes_from_first_provider_note_to_first_ct")
    diff_minutes("last_provider_note_time", "first_provider_note_time", "provider_note_span_minutes")

    if "ArrivalInstant" in encounter_master.columns:
        encounter_master["arrival_year"] = encounter_master["ArrivalInstant"].dt.year.astype("Int64")
        encounter_master["arrival_month"] = encounter_master["ArrivalInstant"].dt.month.astype("Int64")
        encounter_master["arrival_day"] = encounter_master["ArrivalInstant"].dt.day.astype("Int64")
        encounter_master["arrival_hour"] = encounter_master["ArrivalInstant"].dt.hour.astype("Int64")
        encounter_master["arrival_weekday"] = encounter_master["ArrivalInstant"].dt.day_name().astype("string")
        encounter_master["arrival_is_weekend"] = encounter_master["ArrivalInstant"].dt.dayofweek.isin([5, 6]).astype("Int64")

encounter_master["PatientMrn"] = normalize_mrn_series(encounter_master["PatientMrn"])
mrn_counts = (
    encounter_master.dropna(subset=["PatientMrn"])
    .groupby("PatientMrn")
    .size()
    .rename("encounters_for_patient")
    .reset_index()
)
encounter_master = encounter_master.merge(mrn_counts, on="PatientMrn", how="left")
encounter_master["encounters_for_patient"] = encounter_master["encounters_for_patient"].fillna(1).astype("Int64")
encounter_master["same_patient_multiple_encounters"] = (
    encounter_master["encounters_for_patient"] > 1
).astype("Int64")

encounter_master["missing_provider_note"] = (
    encounter_master.get("has_provider_note", 0).fillna(0).astype(int) == 0
).astype("Int64")
encounter_master["missing_triage_note"] = (
    encounter_master.get("has_triage_note", 0).fillna(0).astype(int) == 0
).astype("Int64")
encounter_master["missing_ct_report"] = (
    encounter_master.get("has_ct_report", 0).fillna(0).astype(int) == 0
).astype("Int64")
encounter_master["missing_chief_complaint"] = encounter_master["ChiefComplaint"].isna().astype("Int64")
encounter_master["missing_arrival_time"] = encounter_master["ArrivalInstant"].isna().astype("Int64")
encounter_master["mrn_missing"] = encounter_master["PatientMrn"].isna().astype("Int64")

encounter_master = encounter_master.merge(
    encounter_text_aggregates[[
        "EncounterCsn",
        "all_text_concat_char_count",
        "provider_text_concat_char_count",
        "triage_text_concat_char_count",
        "ct_text_concat_char_count",
        "ct_narrative_concat_char_count",
        "ct_impression_concat_char_count",
    ]].copy(),
    on="EncounterCsn",
    how="left"
)

for c in [
    "all_text_concat_char_count",
    "provider_text_concat_char_count",
    "triage_text_concat_char_count",
    "ct_text_concat_char_count",
    "ct_narrative_concat_char_count",
    "ct_impression_concat_char_count",
]:
    if c in encounter_master.columns:
        encounter_master[c] = encounter_master[c].fillna(0).astype("Int64")

encounter_master["has_ecg_data_placeholder"] = pd.NA
encounter_master["has_ehr_structured_data_placeholder"] = pd.NA

preferred_order = [
    "EncounterCsn", "PatientMrn", "encounters_for_patient", "same_patient_multiple_encounters",
    "ArrivalInstant", "arrival_year", "arrival_month", "arrival_day", "arrival_hour", "arrival_weekday", "arrival_is_weekend",
    "ChiefComplaint", "is_chest_pain", "is_shortness_of_breath",
    "in_chest_pain_sob_cohort", "in_pe_workup_cohort", "pe_suspected",
    "included_in_labeled_200",
    "has_provider_note", "has_triage_note", "has_ct_report",
    "has_any_text", "has_both_provider_and_ct", "has_any_triage_provider_ct",
    "provider_note_count", "provider_note_row_count", "multiple_provider_notes",
    "first_provider_note_time", "last_provider_note_time",
    "provider_note_span_minutes",
    "provider_text_char_count", "provider_max_note_char_count",
    "provider_note_types", "provider_author_types", "provider_services",
    "provider_note_statuses", "provider_source_datasets",
    "triage_note_count", "triage_note_row_count",
    "first_triage_note_time", "last_triage_note_time",
    "triage_text_char_count", "triage_max_note_char_count",
    "triage_note_types", "triage_author_types", "triage_services", "triage_note_statuses",
    "ct_report_count", "ct_pe_protocol_any",
    "first_ct_ordered_time", "first_ct_time", "last_ct_time",
    "ct_study_names", "ct_cpt_codes",
    "ct_narrative_char_count", "ct_impression_char_count", "ct_max_impression_char_count",
    "minutes_from_arrival_to_first_provider_note",
    "minutes_from_arrival_to_first_triage_note",
    "minutes_from_arrival_to_first_ct_order",
    "minutes_from_arrival_to_first_ct",
    "minutes_from_first_provider_note_to_first_ct",
    "all_text_concat_char_count",
    "provider_text_concat_char_count",
    "triage_text_concat_char_count",
    "ct_text_concat_char_count",
    "ct_narrative_concat_char_count",
    "ct_impression_concat_char_count",
    "labeled_200_note_count", "labeled_200_text_char_count",
    "missing_provider_note", "missing_triage_note", "missing_ct_report",
    "missing_chief_complaint", "missing_arrival_time", "mrn_missing",
    "in_ed_triage_provider_sample", "sample_has_triage_note", "sample_has_provider_note", "sample_note_count",
    "has_ecg_data_placeholder", "has_ehr_structured_data_placeholder",
]
preferred_order = [c for c in preferred_order if c in encounter_master.columns]
remaining_cols = [c for c in encounter_master.columns if c not in preferred_order]
encounter_master = encounter_master[preferred_order + remaining_cols].copy()

encounter_master_string_cols = [
    "PatientMrn", "ChiefComplaint", "provider_note_types", "provider_author_types",
    "provider_services", "provider_note_statuses", "provider_source_datasets",
    "triage_note_types", "triage_author_types", "triage_services", "triage_note_statuses",
    "ct_study_names", "ct_cpt_codes", "arrival_weekday",
]
for c in encounter_master_string_cols:
    if c in encounter_master.columns:
        encounter_master[c] = encounter_master[c].astype("string")

encounter_text_aggregates_string_cols = [
    "all_text_concat", "provider_text_concat", "triage_text_concat", "ct_text_concat",
    "ct_narrative_concat", "ct_impression_concat",
    "earliest_provider_note_text", "earliest_triage_note_text", "earliest_ct_text",
]
for c in encounter_text_aggregates_string_cols:
    if c in encounter_text_aggregates.columns:
        encounter_text_aggregates[c] = encounter_text_aggregates[c].astype("string")

# =============================================================================
# 4) BUILD summary_outputs_template.jsonl
# =============================================================================

summary_template_df = encounter_master[[
    "EncounterCsn",
    "PatientMrn",
    "ArrivalInstant",
    "ChiefComplaint",
    "is_chest_pain",
    "is_shortness_of_breath",
    "pe_suspected",
    "included_in_labeled_200",
    "has_provider_note",
    "has_triage_note",
    "has_ct_report",
    "provider_note_count",
    "triage_note_count",
    "ct_report_count",
]].copy()

summary_template_df = summary_template_df.merge(
    encounter_text_aggregates[[
        "EncounterCsn",
        "provider_text_concat",
        "triage_text_concat",
        "ct_narrative_concat",
        "ct_impression_concat",
        "ct_text_concat",
        "all_text_concat",
    ]].copy(),
    on="EncounterCsn",
    how="left"
)

# =============================================================================
# SAVE ALL OUTPUTS
# =============================================================================

encounter_master_path = OUTPUT_DIR / "encounter_master.csv"
notes_raw_path = OUTPUT_DIR / "notes_raw.csv"
encounter_aggregates_path = OUTPUT_DIR / "encounter_text_aggregates.csv"
summary_template_path = OUTPUT_DIR / "summary_outputs_template.jsonl"

encounter_master.to_csv(encounter_master_path, index=False)
notes_raw.to_csv(notes_raw_path, index=False)
encounter_text_aggregates.to_csv(encounter_aggregates_path, index=False)

if WRITE_PARQUET:
    encounter_master_parquet_path = OUTPUT_DIR / "encounter_master.parquet"
    notes_raw_parquet_path = OUTPUT_DIR / "notes_raw.parquet"
    encounter_aggregates_parquet_path = OUTPUT_DIR / "encounter_text_aggregates.parquet"

    encounter_master_safe = make_columns_parquet_safe(encounter_master)
    notes_raw_safe = make_columns_parquet_safe(notes_raw)
    encounter_text_aggregates_safe = make_columns_parquet_safe(encounter_text_aggregates)

    encounter_master_safe.to_parquet(encounter_master_parquet_path, index=False)
    notes_raw_safe.to_parquet(notes_raw_parquet_path, index=False)
    encounter_text_aggregates_safe.to_parquet(encounter_aggregates_parquet_path, index=False)

with open(summary_template_path, "w", encoding="utf-8") as f:
    for _, row in summary_template_df.iterrows():
        template = {
            "EncounterCsn": None if pd.isna(row["EncounterCsn"]) else int(row["EncounterCsn"]),
            "PatientMrn": None if pd.isna(row["PatientMrn"]) else str(row["PatientMrn"]),
            "ArrivalInstant": None if pd.isna(row["ArrivalInstant"]) else str(row["ArrivalInstant"]),
            "ChiefComplaint": None if pd.isna(row["ChiefComplaint"]) else str(row["ChiefComplaint"]),
            "is_chest_pain": None if pd.isna(row["is_chest_pain"]) else int(row["is_chest_pain"]),
            "is_shortness_of_breath": None if pd.isna(row["is_shortness_of_breath"]) else int(row["is_shortness_of_breath"]),
            "pe_suspected": None if pd.isna(row["pe_suspected"]) else int(row["pe_suspected"]),
            "included_in_labeled_200": None if pd.isna(row["included_in_labeled_200"]) else int(row["included_in_labeled_200"]),
            "has_provider_note": None if pd.isna(row["has_provider_note"]) else int(row["has_provider_note"]),
            "has_triage_note": None if pd.isna(row["has_triage_note"]) else int(row["has_triage_note"]),
            "has_ct_report": None if pd.isna(row["has_ct_report"]) else int(row["has_ct_report"]),
            "provider_note_count": None if pd.isna(row["provider_note_count"]) else int(row["provider_note_count"]),
            "triage_note_count": None if pd.isna(row["triage_note_count"]) else int(row["triage_note_count"]),
            "ct_report_count": None if pd.isna(row["ct_report_count"]) else int(row["ct_report_count"]),
            "input_text": {
                "provider_text_concat": None if pd.isna(row.get("provider_text_concat")) else str(row.get("provider_text_concat")),
                "triage_text_concat": None if pd.isna(row.get("triage_text_concat")) else str(row.get("triage_text_concat")),
                "ct_narrative_concat": None if pd.isna(row.get("ct_narrative_concat")) else str(row.get("ct_narrative_concat")),
                "ct_impression_concat": None if pd.isna(row.get("ct_impression_concat")) else str(row.get("ct_impression_concat")),
                "ct_text_concat": None if pd.isna(row.get("ct_text_concat")) else str(row.get("ct_text_concat")),
                "all_text_concat": None if pd.isna(row.get("all_text_concat")) else str(row.get("all_text_concat")),
            },
            "summary_schema_version": "v1",
            "guideline_bundle_used": None,
            "guideline_version": None,
            "prompt_version": None,
            "model_name": None,
            "model_version": None,
            "created_at": None,
            "summary_output": {
                "one_line_case_summary": None,
                "evidence_for_pe": [],
                "evidence_against_pe_or_alternative_diagnoses": [],
                "alternative_diagnoses": [],
                "missing_key_history": [],
                "missing_key_exam_or_workup": [],
                "immediate_next_diagnostic_considerations": [],
                "teaching_note_for_residents": None,
                "confidence_uncertainty_statement": None,
                "confidence_score": None,
            },
        }
        f.write(json.dumps(template, ensure_ascii=False) + "\n")

# =============================================================================
# OPTIONAL QA PRINTS
# =============================================================================

print("\n" + "=" * 100)
print("BUILD COMPLETE")
print("=" * 100)
print(f"encounter_master csv:          {encounter_master.shape} -> {encounter_master_path}")
print(f"notes_raw csv:                 {notes_raw.shape} -> {notes_raw_path}")
print(f"encounter_text_aggregates csv: {encounter_text_aggregates.shape} -> {encounter_aggregates_path}")
print(f"summary_outputs_template:      {len(summary_template_df):,} JSONL rows -> {summary_template_path}")

if WRITE_PARQUET:
    print(f"encounter_master parquet:      {encounter_master.shape} -> {encounter_master_parquet_path}")
    print(f"notes_raw parquet:             {notes_raw.shape} -> {notes_raw_parquet_path}")
    print(f"encounter_aggregates parquet:  {encounter_text_aggregates.shape} -> {encounter_aggregates_parquet_path}")

print("\nHigh-level counts:")
print(f"Unique encounters in encounter_master: {encounter_master['EncounterCsn'].nunique():,}")
print(f"PE workup / suspected PE encounters: {int(encounter_master['pe_suspected'].fillna(0).sum()):,}")
print(f"Chest pain/SOB cohort: {int(encounter_master['in_chest_pain_sob_cohort'].fillna(0).sum()):,}")
print(f"Included in labeled 200: {int(encounter_master['included_in_labeled_200'].fillna(0).sum()):,}")
print(f"Has provider note: {int(encounter_master['has_provider_note'].fillna(0).sum()):,}")
print(f"Has triage note: {int(encounter_master['has_triage_note'].fillna(0).sum()):,}")
print(f"Has CT report: {int(encounter_master['has_ct_report'].fillna(0).sum()):,}")

if sample_enc_summary is not None and "in_ed_triage_provider_sample" in encounter_master.columns:
    print(f"In ED triage/provider sample overlap: {int(encounter_master['in_ed_triage_provider_sample'].fillna(0).sum()):,}")

print("\nNote source counts from standardized note inputs:")
print(notes_all["note_source"].value_counts(dropna=False))

print("\nA few useful QA sums:")
qa_cols = [c for c in [
    "pe_suspected",
    "in_chest_pain_sob_cohort",
    "has_provider_note",
    "has_triage_note",
    "has_ct_report",
    "included_in_labeled_200"
] if c in encounter_master.columns]
print(encounter_master[qa_cols].fillna(0).astype(int).sum())

print("\nDone.")

Reading: medic-saf-pe-cases-mich-med-2023-2025-case-list.csv
Reading: mich-med-chest-pain-sob-visits-2023-2025-case-list.csv
Reading: medic-saf-pe-cases-mich-med-2023-2025-case-ed-provider-notes.csv
Reading: mich-med-chest-pain-sob-visits-2023-2025-case-ed-provider-and-triage-notes.csv
Reading: mich-med-ct-chest-2023-2025-reports.csv
Reading: notes-for-200-cases.csv
Reading: ED-triage-and-provider-sample.csv


/tmp/ipykernel_1334038/1280607537.py:120: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out = out.fillna(df[c])
/tmp/ipykernel_1334038/1280607537.py:120: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out = out.fillna(df[c])
/tmp/ipykernel_1334038/1280607537.py:120: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out = out.fillna(df[c])
/tmp


BUILD COMPLETE
encounter_master csv:          (45036, 95) -> /nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes/derived_tables/encounter_master.csv
notes_raw csv:                 (133958, 32) -> /nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes/derived_tables/notes_raw.csv
encounter_text_aggregates csv: (44570, 16) -> /nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes/derived_tables/encounter_text_aggregates.csv
summary_outputs_template:      45,036 JSONL rows -> /nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes/derived_tables/summary_outputs_template.jsonl
encounter_master parquet:      (45036, 95) -> /nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes/derived_tables/encounter_master.parquet
notes_raw parquet:             (133958, 32) -> /nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes/derived_tables/notes_raw.parquet
encounter_aggregates 

In [12]:
import pandas as pd

master = pd.read_parquet("/nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes/derived_tables/encounter_master.parquet")
notes = pd.read_parquet("/nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes/derived_tables/notes_raw.parquet")
agg = pd.read_parquet("/nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes/derived_tables/encounter_text_aggregates.parquet")

print(master.shape)
print(notes.shape)
print(agg.shape)
print(notes["note_id"].dtype)
print(notes["note_source"].value_counts(dropna=False))

(45036, 95)
(133958, 32)
(44570, 16)
string
note_source
provider     56686
ct_report    48061
triage       29211
Name: count, dtype: Int64


In [13]:
import pandas as pd

df = pd.read_parquet(
    "/nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes/derived_tables/encounter_master.parquet"
)

cols = [
    "pe_suspected",
    "in_chest_pain_sob_cohort",
    "has_triage_note",
    "has_provider_note",
    "has_ct_report",
]

summary = (
    df.groupby(cols, dropna=False)
      .size()
      .reset_index(name="n")
      .sort_values("n", ascending=False)
)

print(summary)

    pe_suspected  in_chest_pain_sob_cohort  has_triage_note  has_provider_note  has_ct_report      n
6              0                         1                1                  1              0  20386
11             1                         0                0                  1              1  14019
16             1                         1                1                  1              1   6876
1              0                         0                0                  0              1   1618
4              0                         1                1                  0              0   1216
2              0                         1                0                  0              0    296
0              0                         0                0                  0              0    169
3              0                         1                0                  1              0    129
7              0                         1                1                  1             

In [14]:
df[
    (df.pe_suspected == 1) &
    (df.in_chest_pain_sob_cohort == 1) &
    (df.has_triage_note == 1) &
    (df.has_provider_note == 1) &
    (df.has_ct_report == 1)
].shape

(6876, 95)